In [0]:
%sql
-- ΕΝΟΤΗΤΑ 1: Αρχικοποίηση Περιβάλλοντος και Δημιουργία Σχημάτων (Medallion Setup)
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "bronze_marios_urban";
CREATE WIDGET TEXT silver_schema DEFAULT "silver_marios_urban";
CREATE WIDGET TEXT gold_schema DEFAULT "gold_marios_urban";
CREATE WIDGET TEXT volume_landing_path DEFAULT "/Volumes/bootcamp/bronze_marios_urban/landing";

In [0]:
%sql
--Phase 3 · Deliverable: Pipeline Validation Checks (assert_true)

-- 1. Bronze lines - rejected lines = fact rows
SELECT assert_true(
  (SELECT count(*) FROM bootcamp.bronze_marios_urban.brz_order_items) 
  - (SELECT count(*) FROM bootcamp.silver_marios_urban.etl_rejects WHERE source_table = 'brz_order_items') 
  = (SELECT count(*) FROM bootcamp.gold_marios_urban.fact_sales),
  'Check 1 Failed: Bronze lines minus rejected lines != fact rows'
) AS check_row_reconciliation;




In [0]:
%sql
-- 2. Σ net_amount in silver = Σ net_amount in gold
SELECT assert_true(
  abs((SELECT round(sum(net_amount), 2) FROM bootcamp.silver_marios_urban.slv_order_items) 
    - (SELECT round(sum(net_amount), 2) FROM bootcamp.gold_marios_urban.fact_sales)) < 0.01,
  'Check 2 Failed: Sum of net_amount mismatch between silver and gold'
) AS check_net_amount_reconciliation;

In [0]:
%sql
-- 3. No duplicate primary keys
SELECT assert_true(
  count(*) = count(DISTINCT sales_key),
  'Check 4 Failed: Duplicate sales_key found in fact_sales'
) AS check_no_duplicate_pks
FROM bootcamp.gold_marios_urban.fact_sales;

In [0]:
%sql
/*
  Q1: What are monthly revenue, gross margin and units sold, by region and store?
  Gold tables: fact_sales, dim_date, dim_store
*/
SELECT 
  d.year,
  d.month,
  d.month_name,
  s.region,
  s.store_name,
  sum(f.quantity) AS units_sold,
  round(sum(f.net_amount), 2) AS monthly_revenue,
  round(sum(f.margin_amount), 2) AS gross_margin,
  round((sum(f.margin_amount) / nullif(sum(f.net_amount), 0)) * 100, 2) AS margin_pct
FROM bootcamp.gold_marios_urban.fact_sales f
JOIN bootcamp.gold_marios_urban.dim_date d 
  ON f.order_date_key = d.date_key
JOIN bootcamp.gold_marios_urban.dim_store s 
  ON f.store_key = s.store_key
GROUP BY d.year, d.month, d.month_name, s.region, s.store_name
ORDER BY d.year, d.month, monthly_revenue DESC;

In [0]:
%sql
/*
  Q2: Which product categories and brands drive the most margin?
  Gold tables: fact_sales, dim_product
*/
SELECT 
  p.category,
  p.brand,
  round(sum(f.net_amount), 2) AS total_revenue,
  round(sum(f.cost_amount), 2) AS total_cost,
  round(sum(f.margin_amount), 2) AS total_margin,
  round((sum(f.margin_amount) / nullif(sum(f.net_amount), 0)) * 100, 2) AS margin_pct
FROM bootcamp.gold_marios_urban.fact_sales f
JOIN bootcamp.gold_marios_urban.dim_product p 
  ON f.product_key = p.product_key
WHERE f.product_key != -1
GROUP BY p.category, p.brand
ORDER BY total_margin DESC;

In [0]:
%sql
/*
  Q3: Who are our top customers per segment?
  Gold tables: fact_sales, dim_customer
*/
WITH customer_spending AS (
  SELECT 
    c.segment,
    c.customer_id,
    concat(c.first_name, ' ', c.last_name) AS customer_name,
    c.email,
    count(DISTINCT f.order_id) AS total_orders,
    round(sum(f.net_amount), 2) AS total_spend,
    round(sum(f.margin_amount), 2) AS customer_profit
  FROM bootcamp.gold_marios_urban.fact_sales f
  JOIN bootcamp.gold_marios_urban.dim_customer c 
    ON f.customer_key = c.customer_key
  WHERE f.customer_key != -1
  GROUP BY c.segment, c.customer_id, c.first_name, c.last_name, c.email
),
ranked_customers AS (
  SELECT 
    *,
    dense_rank() OVER (PARTITION BY segment ORDER BY total_spend DESC) AS segment_rank
  FROM customer_spending
)
SELECT *
FROM ranked_customers
WHERE segment_rank <= 5
ORDER BY segment, segment_rank;

In [0]:
%sql
/*
  Q4: What is the return rate by category and store type?
  Gold tables: fact_sales, dim_product, dim_store
*/
SELECT 
  p.category,
  s.store_type,
  count(*) AS total_order_lines,
  count(CASE WHEN f.order_status = 'Returned' THEN 1 END) AS returned_order_lines,
  round((count(CASE WHEN f.order_status = 'Returned' THEN 1 END) * 100.0) / count(*), 2) AS return_rate_pct,
  round(sum(CASE WHEN f.order_status = 'Returned' THEN f.net_amount ELSE 0 END), 2) AS returned_net_value
FROM bootcamp.gold_marios_urban.fact_sales f
JOIN bootcamp.gold_marios_urban.dim_product p 
  ON f.product_key = p.product_key
JOIN bootcamp.gold_marios_urban.dim_store s 
  ON f.store_key = s.store_key
WHERE f.product_key != -1
GROUP BY p.category, s.store_type
ORDER BY p.category, return_rate_pct DESC;

In [0]:
%sql
/*
  Q5: How do weekdays vs weekends and online vs physical stores compare?
  Gold tables: fact_sales, dim_date, dim_store
*/
SELECT 
  CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
  s.store_type,
  count(DISTINCT f.order_id) AS total_orders,
  sum(f.quantity) AS total_units_sold,
  round(sum(f.net_amount), 2) AS total_revenue,
  round(sum(f.margin_amount), 2) AS total_margin,
  round(sum(f.net_amount) / nullif(count(DISTINCT f.order_id), 0), 2) AS avg_order_value
FROM bootcamp.gold_marios_urban.fact_sales f
JOIN bootcamp.gold_marios_urban.dim_date d 
  ON f.order_date_key = d.date_key
JOIN bootcamp.gold_marios_urban.dim_store s 
  ON f.store_key = s.store_key
GROUP BY 
  CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END,
  s.store_type
ORDER BY day_type, total_revenue DESC;

In [0]:
%sql
/*
  Phase 6 · Delta Lake Audit Trail
  Επισκόπηση των εκδόσεων (versions), λειτουργιών (operations) και χρονικών στιγμών του πίνακα.
*/
DESCRIBE HISTORY bootcamp.silver_marios_urban.slv_customers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-25T17:38:13.000Z,148255934622515,ey26-004@zone01.gr,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f137cf0a-4738-49ae-926b-9848bb28540c"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e64eb6fb-1b6f-466d-8ee2-e37074b020cb"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-96e24910-5ca1-4062-a0c1-7fa8b94c0b96""}, statsOnLoad -> true)","List(96017058121416, UrbanMart_Job_MariosKatsanevakis, 824293119730446, 767657468956103, 148255934622515, manual, 7405608981153757)",List(980441939303047),960f20cb-aa48-487c-89cf-71612626f5be,1025-173658-upjxu1bc-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 10569, numDeletionVectorsRemoved -> 0, numOutputRows -> 485, numOutputBytes -> 10621)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
1,2026-10-25T16:53:06.000Z,148255934622515,ey26-004@zone01.gr,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f137cf0a-4738-49ae-926b-9848bb28540c"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e64eb6fb-1b6f-466d-8ee2-e37074b020cb"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-96e24910-5ca1-4062-a0c1-7fa8b94c0b96""}, statsOnLoad -> true)",null,List(2542784661074511),60e0920c-e1a5-4292-9500-b5099c761371,1025-093809-agvr4gav-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 10562, numDeletionVectorsRemoved -> 0, numOutputRows -> 485, numOutputBytes -> 10569)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
0,2026-10-20T17:28:50.000Z,148255934622515,ey26-004@zone01.gr,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f137cf0a-4738-49ae-926b-9848bb28540c"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e64eb6fb-1b6f-466d-8ee2-e37074b020cb"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-96e24910-5ca1-4062-a0c1-7fa8b94c0b96""}, statsOnLoad -> true)",null,List(2542784661074511),5caf6967-02ba-4e73-a8df-971d96fbf3cc,1020-064701-2p4c5k1j-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 485, numOutputByte

In [0]:
%sql
/*
  Phase 6 · Time Travel
  Ανάκτηση δεδομένων από προηγούμενο version (version 0).
*/
SELECT count(*) AS count_v0 
FROM bootcamp.silver_marios_urban.slv_customers VERSION AS OF 0;

count_v0
485
